# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

In [2]:
# Directory paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")

# Dates and locations
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "06-19-2026"
prev_end_date = "06-05-2026"
date_range = start_date + "--" + end_date
prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13"] # , "D1.1", "Not"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
prev_downloads_saved = home + "NCBI_Virus/downloads/" + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 

andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

In [3]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0:
        break 
    else: # If we don't have any downloaded files
        # Get files
        # open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [4]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Assembly, as_index=False).size()
print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Assembly")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_counted.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments = metadata_complete_segs
metadata_segments

162538
              Assembly  size
0      GCA_038163845.1     8
1      GCA_038163995.1     8
2      GCA_038164135.1     8
3      GCA_038164315.1     8
4      GCA_038164335.1     8
...                ...   ...
18712  GCA_058182645.1     8
18713  GCA_058315875.1     8
18714  GCA_058329715.1     8
18715  GCA_058329825.1     8
18716  GCA_058329855.1     8

[18717 rows x 2 columns]


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PZ544326.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8
1,PZ544327.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8
2,PZ544328.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8
3,PZ544329.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8
4,PZ544330.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149713,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
149714,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
149715,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
149716,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [5]:
# NCBI Virus Naming Convention:
# "Accession|GenBank_Title|Assembly|SRA Accession|BioSample|BioProject|Genotype|Isolate|Geo Location|Host|Collection Date"

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Extract segment number so that we can add the correct sequences to the correct sample
# sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
# print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PZ437028.1 |Influenza A virus (A/Leptonychote...
1         >PZ544326.1 |Influenza A virus (A/cattle/ID/26...
2         >PZ544327.1 |Influenza A virus (A/cattle/ID/26...
3         >PZ544328.1 |Influenza A virus (A/cattle/ID/26...
4         >PZ544329.1 |Influenza A virus (A/cattle/ID/26...
                                ...                        
162533    >OK205883.1 |Influenza A virus (A/chicken/Vera...
162534    >OK205884.1 |Influenza A virus (A/chicken/Vera...
162535    >OK205885.1 |Influenza A virus (A/chicken/Vera...
162536    >OK205886.1 |Influenza A virus (A/chicken/Vera...
162537    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 162538, dtype: object
162538
148206


In [6]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PZ544326.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8,>PZ544326.1 |Influenza A virus (A/cattle/ID/26...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...
1,PZ544327.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8,>PZ544327.1 |Influenza A virus (A/cattle/ID/26...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...
2,PZ544328.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8,>PZ544328.1 |Influenza A virus (A/cattle/ID/26...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...
3,PZ544329.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8,>PZ544329.1 |Influenza A virus (A/cattle/ID/26...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
4,PZ544330.1,GenBank,GCA_058329715.1,SRR38788396,SAMN60347461,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2026-05-04,2026-06-17,ssRNA(-),8,>PZ544330.1 |Influenza A virus (A/cattle/ID/26...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148201,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205883.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
148202,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205884.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
148203,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205885.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
148204,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205886.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


## Find genotypes

(Using old genoflu results or Andersen Lab genoflu output) <br>
Old genoflu results should accumulate into one file to avoid having to genotype anything again.

### Find old genotypes

In [7]:
# Switch directory to previous week
os.chdir(prev_downloads_saved)

# Get both files from previous week
genoflu_output = pd.read_csv("output.tsv", delimiter="\t")
genoflu_results = pd.read_csv("results.tsv", delimiter="\t")

# Get old results from output.tsv
genoflu_old = pd.concat([genoflu_output, genoflu_results])
# genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"}) # So we can merge
print(genoflu_old)

# Switch back directory
os.chdir(downloads_saved)

# Save this output for the future
genoflu_old.to_csv("output.tsv", sep="\t", index=False)

                                                  Strain  \
0      Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
1      Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
2      Influenza_A_virus__Mexico__Ciudad_de_Mexico_CP...   
3      Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
4      Influenza_A_virus__Mexico__Michoacan_CPA_02011...   
...                                                  ...   
13011                                    GCA_039460795_1   
13012                                    GCA_046427615_1   
13013                                    GCA_048675505_1   
13014                                    GCA_039288985_1   
13015                                    GCA_047522105_1   

                                                Genotype  \
0      Not assigned: Only 3 segments >98.0% match fou...   
1      Not assigned: Only 3 segments >98.0% match fou...   
2      Not assigned: Only 3 segments >98.0% match fou...   
3      Not assigned: Only 3 segments >9

In [8]:

metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

# Merge to get already-genotyped segments
genoflu_old["Partial_Header_temp"] = genoflu_old["Strain"].apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1]) # Get only isolate 
genoflu_old["Partial_Header_temp"] = genoflu_old["Partial_Header_temp"].apply(lambda x: re.split(r'_H.N._20.{2}_.{2}_.{2}', x)[0]) # Get only isolate

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header_temp") 

# Isolate not-already-genotyped segments
metadata_segments_new = metadata_segments.merge(genoflu_old, indicator=True, how='left', on="Partial_Header_temp").loc[lambda x : x['_merge']=='left_only'] 

print(len(metadata_segments_old))
print(len(metadata_segments_new))

128
148078


In [9]:
# Get genoflu results from Andersen and see if any fit

os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"}) # So we can merge with old
# genoflu_andersen["SRA_Accession"] = genoflu_andersen["Strain"] # So we can merge with new

# Find those genotyped by Andersen via merge
metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
# Rename genotype by Andersen to concatenate
metadata_segments_known_andersen["Genotype_y"] = metadata_segments_known_andersen["Genotype"]

# Keep all known genotypes
metadata_segments_known = pd.concat([metadata_segments_old, metadata_segments_known_andersen]) # , on="SRA_Accession", how="left") # Since we know both of these

print(metadata_segments_old)
print(metadata_segments_known_andersen)

        Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0      PZ544326.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
1      PZ544327.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
2      PZ544328.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
3      PZ544329.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
4      PZ544330.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
...           ...            ...              ...           ...           ...   
81315  PP740725.1        GenBank  GCA_039251605.1   SRR28752446  SAMN41019184   
81316  PP740726.1        GenBank  GCA_039251605.1   SRR28752446  SAMN41019184   
81317  PP740727.1        GenBank  GCA_039251605.1   SRR28752446  SAMN41019184   
81318  PP740728.1        GenBank  GCA_039251605.1   SRR28752446  SAMN41019184   
81319  PP740729.1        GenBank  GCA_039251605.1   SRR28752446  SAMN41019184   

         BioProject      Or

### Create FASTA files of unknown genotypes 

In [10]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Average Depth of Coverage List_x,_merge,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y
0,OL539626.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OL539627.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OL539628.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OL539629.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OL539630.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81315,PP740725.1,GenBank,GCA_039251605.1,SRR28752446,SAMN41019184,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report
81316,PP740726.1,GenBank,GCA_039251605.1,SRR28752446,SAMN41019184,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report
81317,PP740727.1,GenBank,GCA_039251605.1,SRR28752446,SAMN41019184,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report
81318,PP740728.1,GenBank,GCA_039251605.1,SRR28752446,SAMN41019184,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report


In [11]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_new["Partial_Header"] = metadata_segments_new["Assembly"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name
metadata_segments_new = metadata_segments_new.dropna(subset="Partial_Header")

print(metadata_segments_new["Partial_Header"].values[0:5])

['>GCA_058329715.1' '>GCA_058329715.1' '>GCA_058329715.1'
 '>GCA_058329715.1' '>GCA_058329715.1']


In [12]:


# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_new["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_new[metadata_segments_new["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_new["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

KeyboardInterrupt: 

### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [12]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [13]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Assembly"] 
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# Merge
metadata_genoflu = metadata_segments_new.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 

# # Fill the rest of the 8 segments with the same genotype
# metadata_genoflu = metadata_genoflu.ffill(limit_area="inside", limit=7)



print(metadata_genoflu)


     Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0   PZ544326.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
1   PZ544327.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
2   PZ544328.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
3   PZ544329.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
4   PZ544330.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
5   PZ544331.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
6   PZ544332.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
7   PZ544333.1        GenBank  GCA_058329715.1   SRR38788396  SAMN60347461   
8   PZ544334.1        GenBank  GCA_058329825.1   SRR38788385  SAMN60347462   
9   PZ544335.1        GenBank  GCA_058329825.1   SRR38788385  SAMN60347462   
10  PZ544336.1        GenBank  GCA_058329825.1   SRR38788385  SAMN60347462   
11  PZ544337.1        GenBank  GCA_058329825.1   SRR38788385  SA

### Concatenate with known genotypes

In [14]:
# Rename columns so we can concatenate
print(metadata_segments_known.columns)

metadata_segments_known = metadata_segments_known.reset_index(drop=True)
print(metadata_segments_known)

print(metadata_genoflu.columns)
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

metadata_genoflu = metadata_genoflu.reset_index(drop=True)
print(metadata_genoflu)

# Concatenation
metadata_genoflu_concat = pd.concat([metadata_segments_known, metadata_genoflu])

metadata_genoflu_concat

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Partial_Header_temp', 'Strain',
       'Genotype_y', 'Genotype List Used, >=98.0%',
       'Genotype Sample Title List', 'Genotype Percent Match List',
       'Genotype Mismatch List', 'Genotype Average Depth of Coverage List',
       'Date run', 'Genotype List Used, >=98.0%_x',
       'Genotype Sample Title List_x', 'Genotype Percent Match List_x',
       'Genotype Mismatch List_x', 'Genotype Average Depth of Coverage List_x',
       '_merge', 'date', 'File Name', 'Genotype',
       'Genotype List Used, >=98.0

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Genotype_official,Serotype,Strain_x,Date run_x,Partial_Header,Partial_Header_Merge,Strain_y,Date run_y
0,OL539626.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N3,NaN,NaN,NaN,NaN,NaN,NaN
1,OL539627.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N3,NaN,NaN,NaN,NaN,NaN,NaN
2,OL539628.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N3,NaN,NaN,NaN,NaN,NaN,NaN
3,OL539629.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N3,NaN,NaN,NaN,NaN,NaN,NaN
4,OL539630.1,GenBank,GCA_039236455.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N3,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,PQ573561.1,GenBank,GCA_046435525.1,NaN,NaN,PRJNA1095491,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"1, 17, 2, 7, 1, 0, 1, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_046435525.1,GCA_046435525_1,GCA_046435525_1,2026-06-25_12-24-08
52,PQ573562.1,GenBank,GCA_046435525.1,NaN,NaN,PRJNA1095491,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"1, 17, 2, 7, 1, 0, 1, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_046435525.1,GCA_046435525_1,GCA_046435525_1,2026-06-25_12-24-08
53,PQ573563.1,GenBank,GCA_046435525.1,NaN,NaN,PRJNA1095491,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"1, 17, 2, 7, 1, 0, 1, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_046435525.1,GCA_046435525_1,GCA_046435525_1,2026-06-25_12-24-08
54,PQ573564.1,GenBank,GCA_046435525.1,NaN,NaN,PRJNA1095491,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"1, 17, 2, 7, 1, 0, 1, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_046435525.1,GCA_046435525_1,GCA_046435525_1,2026-06-25_12-24-08


In [27]:
# Cut down to only columns we want
metadata_genoflu_concat = metadata_genoflu_concat[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header"]] #, "Strain"]]

# Get genbank strain name
metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu_concat = metadata_genoflu_concat.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_90640\4267596827.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_90640\4267596827.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

In [28]:
metadata_genoflu_concat

# Make sure we only have the serotype(s) we want
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [29]:
# Re-re-name so that we don't break following code

metadata_genoflu = metadata_genoflu_concat

After running the below code, **STOP TO CHECK** if any new animals appear

In [30]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['hermit thrush', 'western sandpiper', 'peregrine falcon', 'pekin duck', 'cackling goose', 'cattle', 'mink', 'wild duck', 'ruddy duck', 'heron', 'skunk', 'rough-legged hawk', 'burrowing owl', 'domestic cat', 'fish crow', 'bluejay', 'crow', 'red-shouldered hawk', 'snowy egret', 'lynx', 'house mouse', 'pelican', 'common eider', 'lesser black-backed gull', 'mallard duck', 'hooded merganser', 'harbor seal', 'goat', 'northern elephant seal', 'glaucous-winged gull', 'common merganser', 'ring-billed gull', 'serval', 'bobcat', 'rock dove', 'western gull', 'blue-winged teal', 'great egret', 'dove', 'pig', 'mute swan', 'lesser snow goose', 'lion', 'common murre', 'canada goose', 'chinese goose', 'glaucous gull', 'american green-winged teal', 'common goldeneye', 'rock pigeon', 'owl', 'wood duck', 'great black-backed gull', 'fox', 'black swan', 'mottled duck', 'greater white-fronted goose', 'laughing gull', 'pigeon', 'california gull', 'chukar', 'european starling', 'sanderling', 'snowy owl', 'com

In [31]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [32]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [33]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,genbank_name,Genotype
0,OL539626.1,GCA_039236455.1,Influenza A virus (A/mallard/New York/AH017938...,mallard,2021-09-03,NaN,AH0179389,Not,"USA: Steuben County, New York",>OL539626.1 |Influenza A virus (A/mallard/New ...,ATGGAGAGAATAAAAGAACTAAGAGATCTAATGTCACAGTCTCGCA...,H5N3,1,NaN,NaN,A/mallard/New York/AH0179389/2021,Not
1,OL539627.1,GCA_039236455.1,Influenza A virus (A/mallard/New York/AH017938...,mallard,2021-09-03,NaN,AH0179389,Not,"USA: Steuben County, New York",>OL539627.1 |Influenza A virus (A/mallard/New ...,ATGGATGTCAATCCGACTTTACTCTTCTTGAAAGTTCCAGCGCAAA...,H5N3,2,NaN,NaN,A/mallard/New York/AH0179389/2021,Not
2,OL539628.1,GCA_039236455.1,Influenza A virus (A/mallard/New York/AH017938...,mallard,2021-09-03,NaN,AH0179389,Not,"USA: Steuben County, New York",>OL539628.1 |Influenza A virus (A/mallard/New ...,ATGGAAGATTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...,H5N3,3,NaN,NaN,A/mallard/New York/AH0179389/2021,Not
3,OL539629.1,GCA_039236455.1,Influenza A virus (A/mallard/New York/AH017938...,mallard,2021-09-03,NaN,AH0179389,Not,"USA: Steuben County, New York",>OL539629.1 |Influenza A virus (A/mallard/New ...,ATGGAAAGAATAGTGATTGCCCTCGCAATAATCAGCATTGTCAAAG...,H5N3,4,NaN,NaN,A/mallard/New York/AH0179389/2021,Not
4,OL539630.1,GCA_039236455.1,Influenza A virus (A/mallard/New York/AH017938...,mallard,2021-09-03,NaN,AH0179389,Not,"USA: Steuben County, New York",>OL539630.1 |Influenza A virus (A/mallard/New ...,ATGGCGTCTCAAGGCACCAAACGATCTTATGAACAAATGGAAACTG...,H5N3,5,NaN,NaN,A/mallard/New York/AH0179389/2021,Not
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,PQ573561.1,GCA_046435525.1,Influenza A virus (A/Washington/240/2024(H5N1)...,Washington,2024-10-18,NaN,A/Washington/240/2024,D1.1,USA: Washington,>PQ573561.1 |Influenza A virus (A/Washington/2...,TCAAATATATTCAATATGGAGAGAATAAAAGAACTGAGAGATCTAA...,H5N1,1,NaN,>GCA_046435525.1,A/Washington/240/2024,D1.1
52,PQ573562.1,GCA_046435525.1,Influenza A virus (A/Washington/240/2024(H5N1)...,Washington,2024-10-18,NaN,A/Washington/240/2024,D1.1,USA: Washington,>PQ573562.1 |Influenza A virus (A/Washington/2...,GGTTCACTCTGTCAAAATGGAAAACATAGTACTTCTTCTTGCAATA...,H5N1,4,NaN,>GCA_046435525.1,A/Washington/240/2024,D1.1
53,PQ573563.1,GCA_046435525.1,Influenza A virus (A/Washington/240/2024(H5N1)...,Washington,2024-10-18,NaN,A/Washington/240/2024,D1.1,USA: Washington,>PQ573563.1 |Influenza A virus (A/Washington/2...,AGTTCAAAATGAATCCAAATCAAAAGATAATAACTATCGGGTCAAT...,H5N1,6,NaN,>GCA_046435525.1,A/Washington/240/2024,D1.1
54,PQ573564.1,GCA_046435525.1,Influenza A virus (A/Washington/240/2024(H5N1)...,Washington,2024-10-18,NaN,A/Washington/240/2024,D1.1,USA: Washington,>PQ573564.1 |Influenza A virus (A/Washington/2...,GTAGATAATCACTCACTGAGTGACATCCACATCATGGCGTCTCAAG...,H5N1,5,NaN,>GCA_046435525.1,A/Washington/240/2024,D1.1


In [115]:
def geo_location_normalize(geolocation: str) -> str:

    geolocation = geolocation.replace(":", ",") # We will need to split on commas later; e.g. USA: MD -> USA, MD

    country = geolocation.split(",")[0] # Get the first part of the geolocation, aka the country 

    state = geolocation.split(",")[-1] # Get the last part of the geolocation, aka the state

    state = state.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country = country.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country_state = country + "-" + state # Log needed

    return country_state 

# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

      
    metadata["Geo_Location_State_USA"] = metadata["Geo_Location_State_med"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name
    else x)
    

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    return metadata

def geo_location_strain_name(metadata, state_ref_file):
    # If there is no state in the metadata, try the strain name
    state_ref = pd.read_csv(state_ref_file)

    metadata["strain_name_state_nonhuman"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2])

    metadata["strain_name_state_human"] = metadata["genbank_name"].apply(lambda x: x.split("/")[1])

    # If nonhuman, do strain name [2]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] != "human"), metadata["strain_name_state_nonhuman"], metadata["Geo_Location_State_USA"])

    # If human, do strain name [1]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] == "human"), metadata["strain_name_state_human"], metadata["Geo_Location_State_USA"])

    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name"].apply(geo_location_normalize)

    metadata["Geo_Location_State_Strain_Name_State"] = metadata["Geo_Location_State_Strain_Name"].apply(lambda x: x.split("-")[-1])
    
    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name_State"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name, and is just USA
    else ""
    if x == "USA" or x == "United_States"
    
    else x)

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_Strain_Name"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata

    
    return metadata

# # Format: USA-[state abbreviation], e.g. USA-MD
# def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

#     state_ref = pd.read_csv(state_ref_file)

#     # Normalize the locations

#     # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

#     metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

#     metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

#     # Get country
#     metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
#                                                                               if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
#                                                                               else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
#                                                                               if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
#                                                                               else x)

#     # Get state
#     metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

#     try:    
#         metadata["Geo_Location_State"] = metadata["Geo_Location_State_med"].apply(lambda x: 
#         # # If "x" is the abbreviated state (e.g. "MD")
#         state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
#         if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
#         # # If "x" is the state name (e.g. "Maryland")
#         else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
#         if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
#         # # If "x" has neither the state abbreviation nor the full state name
#         else x)
#     except:
#         print("Could not find states in reference.")

#     # If there is no state in the metadata, try the strain name



#     metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State"] 

#     # If USA-, delete -
#     metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
#     if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
#     else x)

#     # Log metadata
#     return metadata

# # Format: USA-[state abbreviation], e.g. USA-MD
# def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

#     state_ref = pd.read_csv(state_ref_file)

#     # Normalize the locations

#     # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

#     metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

#     metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

#     # Get country
#     # metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x, regex=True), 'Country'].iloc[0])

#     # Get state
#     metadata["Geo_Location_State"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

#     try:    
#         metadata["Geo_Location_State"] = metadata["Geo_Location_State"].apply(lambda x: 
#         # If "x" is the abbreviated state (e.g. "MD")
#         state_ref[state_ref['Abbreviation'].str.contains(x)]['Abbreviation'].values[0] # .iloc[0]
#         if state_ref["Abbreviation"].str.contains(x).any()
#         # If "x" is the state name (e.g. "Maryland")
#         else state_ref[state_ref['State'].str.contains(x)]['Abbreviation'].values[0] # .iloc[0] 
#         if state_ref["State"].str.contains(x).any() 
#         # If "x" has neither the state abbreviation nor the full state name
#         else x)
#     except:
#         print("Could not find states in reference.")

#     metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State"] 

#     # If USA-, delete -
#     metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
#     if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
#     else x)

#     # Log metadata
#     return metadata

In [116]:
print(states_ref[states_ref['State'].str.contains("New_York")]["Abbreviation"].values[0])

NY


In [117]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# # Get geographic locations
# metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
#                                                                         # If "x" has the state abbreviation (e.g. "MD")
#                                                                         states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
#                                                                         + "-" + 
#                                                                         x.split(" ")[-1]
#                                                                         if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
#                                                                         # If "x" has the full state name (e.g. "Maryland")
#                                                                         else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
#                                                                         + "-" + 
#                                                                         states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
#                                                                         if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
#                                                                         # If "x" has neither the state abbreviation nor the full state name nor is "USA"
#                                                                         else 
#                                                                         x
#                                                                         )

# metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


# print(metadata_genoflu["Geo_Location_Abrv"])



In [ ]:
os.chdir(references)
metadata_genoflu = geo_location_get(metadata_genoflu, "states_ref.csv")

In [118]:
metadata_genoflu = geo_location_strain_name(metadata_genoflu, "states_ref.csv")

In [119]:
os.chdir(downloads_saved)
metadata_genoflu.to_csv("metadata_checkpoint.csv")

In [58]:
# If there is no SRA Accession, replace identifier with Assembly -- this part might be duplicating accessions :(
metadata_genoflu["Identifier"] = metadata_genoflu["Assembly"].apply(lambda x: x if metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0] == "" else metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0]) # np.where(metadata_genoflu['SRA_Accession'] != "", metadata_genoflu['SRA_Accession'], metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]))
# # If there is no Assembly, replace identifier with Accession -- only for PB2
# metadata_genoflu["Identifier"] = np.where(metadata_genoflu["SRA_Accession_intermediate"] == "", metadata_genoflu["Accession"].apply(lambda x: metadata_genoflu[metadata_genoflu["Accession"] == x]["Accession"].values[0] if metadata_genoflu[metadata_genoflu["Accession"] == x].loc[:, "Segment"].values[0] == 1 else np.nan), metadata_genoflu["SRA_Accession_intermediate"])
# # Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
# metadata_genoflu.loc[:,"Identifier"] = metadata_genoflu.loc[:, "Identifier"].ffill(limit=7, limit_area="inside")

print((metadata_genoflu[metadata_genoflu["Identifier"].str.contains("SRR")])) # metadata_genoflu[(metadata_genoflu["Identifier"].str.contains("GCA")) | 

      Accession         Assembly  \
128  PZ544326.1  GCA_058329715.1   
129  PZ544327.1  GCA_058329715.1   
130  PZ544328.1  GCA_058329715.1   
131  PZ544329.1  GCA_058329715.1   
132  PZ544330.1  GCA_058329715.1   
..          ...              ...   
27   PZ544353.1  GCA_058315875.1   
28   PZ544354.1  GCA_058315875.1   
29   PZ544355.1  GCA_058315875.1   
30   PZ544356.1  GCA_058315875.1   
31   PZ544357.1  GCA_058315875.1   

                                         GenBank_Title    Host  \
128  Influenza A virus (A/cattle/ID/26G07177-001-or...  cattle   
129  Influenza A virus (A/cattle/ID/26G07177-001-or...  cattle   
130  Influenza A virus (A/cattle/ID/26G07177-001-or...  cattle   
131  Influenza A virus (A/cattle/ID/26G07177-001-or...  cattle   
132  Influenza A virus (A/cattle/ID/26G07177-001-or...  cattle   
..                                                 ...     ...   
27   Influenza A virus (A/cattle/ID/26G07177-004-or...  cattle   
28   Influenza A virus (A/cattle/ID/26G

In [59]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_New"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [60]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


81504
81472


## Rename segments and make complete FASTA files

In [61]:
# Set up segments

# if len(genotypes) > 3: # If we're not doing maintenance only
#     genotypes.append("Not assigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.13_PB2
B3.13_PB1
B3.13_PA
B3.13_HA
B3.13_NP
B3.13_NA
B3.13_MP
B3.13_NS


In [62]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

        Accession         Assembly  \
1104   PZ479596.1  GCA_058019015.1   
1112   PZ479604.1  GCA_058019025.1   
1120   PZ479612.1  GCA_058019055.1   
1128   PZ479620.1  GCA_058019105.1   
1136   PZ479628.1  GCA_058019115.1   
...           ...              ...   
81415  PP756076.1  GCA_039465795.1   
81423  PP756084.1  GCA_039464765.1   
81431  PP756092.1  GCA_039465935.1   
81439  PP756100.1  GCA_039466035.1   
81447  PP740729.1  GCA_039251605.1   

                                           GenBank_Title       Host  \
1104   Influenza A virus (A/cattle/ID/26G07162-001-or...     cattle   
1112   Influenza A virus (A/cattle/ID/26G07162-002-or...     cattle   
1120   Influenza A virus (A/cattle/ID/26G07162-003-or...     cattle   
1128   Influenza A virus (A/cattle/ID/26G07162-004-or...     cattle   
1136   Influenza A virus (A/cattle/ID/26G07162-005-or...     cattle   
...                                                  ...        ...   
81415  Influenza A virus (A/cattle/New Mexico/